# Hydrogen production control with AWAC (reinforcement learning)

This notebook drives hydrogen production with a pre-trained AWAC policy. Given live current feedback, it issues voltage and temperature setpoints to track a target current. It reuses the action/observation structure from the training environment (obs=[current,target], action=[dV,dT]) and keeps the same bounds/delta limits.

**Workflow**
1. (optional) Install RL dependencies (`torch`, `d3rlpy`, `gymnasium`, etc.).
2. Point `policy_path` at the saved AWAC policy (`.pt` TorchScript or `.d3` learnable checkpoint).
3. Implement `read_facility_state` / `send_setpoints` for your lab interface.
4. Start the control loop to drive the real system.
5. If you can import `H2ProductionEnv` and have `hydrogen_data.csv`, run the optional simulation block to sanity check behavior offline.

In [ ]:
# Run once if the environment does not already have the RL packages.
# Remove --quiet if you want to see the build logs.
%pip install --quiet --upgrade torch d3rlpy gymnasium pandas numpy matplotlib


In [ ]:
from __future__ import annotations

import time
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import torch
import d3rlpy

from ORCA.Basic.Optimization import RLSetpointController

VOLT_MIN = 1.2
VOLT_MAX = 5.5
TEMP_MIN_C = 700.0
TEMP_MAX_C = 850.0
MAX_DV_PER_STEP = 0.5
MAX_DT_PER_STEP = 5.0
CONTROL_PERIOD_S = 5.0


In [ ]:
def load_awac_policy(policy_path: str | Path, device: str = "cpu") -> Callable[[np.ndarray], np.ndarray]:
    """Load a saved AWAC policy.

    Supports either d3rlpy `.d3` checkpoints or TorchScript policies saved via
    `awac_agent.save_policy(...)`.
    """
    policy_path = Path(policy_path)
    if not policy_path.exists():
        raise FileNotFoundError(f"Policy file not found: {policy_path}")

    if policy_path.suffix == ".d3":
        agent = d3rlpy.load_learnable(str(policy_path), device=device)

        def predict(obs: np.ndarray) -> np.ndarray:
            obs_arr = np.asarray(obs, dtype=np.float32)
            return agent.predict(obs_arr[None, :])[0]

        return predict

    script = torch.jit.load(str(policy_path), map_location=device)
    script.eval()

    def predict(obs: np.ndarray) -> np.ndarray:
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            act = script(obs_t)
        return act.cpu().numpy()[0]

    return predict


## Facility I/O hooks

Replace the placeholders below with the calls that talk to your lab hardware (e.g., OPC UA/Modbus writes, PyVISA, or a serial/REST client).

In [ ]:
def read_facility_state() -> tuple[float, float, float]:
    """Return (current_A, voltage_V, temperature_C) from the plant.

    Implement this to pull live measurements from your instrumentation layer.
    """
    raise NotImplementedError("Connect this to your facility instrumentation")


def send_setpoints(voltage_v: float, temperature_c: float) -> None:
    """Send voltage and temperature commands to the plant.

    Replace this with code that writes to your controller/PLC.
    """
    print(f"[DRY-RUN] Commanding V={voltage_v:.3f} V, T={temperature_c:.1f} C")


In [ ]:
# Point to your saved policy; update the path if the file lives elsewhere.
policy_path = Path("../awac_h2_production_policy.pt")
policy_predict = load_awac_policy(policy_path, device="cpu")

controller = RLSetpointController(
    policy_fn=policy_predict,
    init_voltage=2.5,
    init_temperature=750.0,
    dv_limit=MAX_DV_PER_STEP,
    dt_limit=MAX_DT_PER_STEP,
    voltage_bounds=(VOLT_MIN, VOLT_MAX),
    temp_bounds=(TEMP_MIN_C, TEMP_MAX_C),
)

# Adjust as needed; can also vary dynamically inside the loop.
target_current = 0.8


In [ ]:
def run_live_control(max_steps: int | None = None, period_s: float = CONTROL_PERIOD_S):
    """Simple synchronous control loop for the real system."""
    step = 0
    log: list[dict] = []
    try:
        while True:
            current_a, measured_voltage, measured_temperature = read_facility_state()
            action_delta, v_cmd, t_cmd, raw_action, entry = controller.step(
                measured_current=current_a,
                target_current=target_current,
                timestamp=time.time(),
            )
            entry.update({
                "measured_voltage": measured_voltage,
                "measured_temperature": measured_temperature,
            })
            log.append(entry)

            send_setpoints(v_cmd, t_cmd)
            print(
                f"step={step:04d} | I={current_a:.3f} A -> dV={action_delta[0]:+.3f} V, dT={action_delta[1]:+.3f} C"
                f" | V_cmd={v_cmd:.3f} V, T_cmd={t_cmd:.1f} C"
            )

            step += 1
            if max_steps is not None and step >= max_steps:
                break
            time.sleep(period_s)
    except KeyboardInterrupt:
        print("Control loop interrupted; holding last command.")

    return pd.DataFrame(log)


# Uncomment to start live control once read/send functions are wired up.
# run_live_control(max_steps=20, period_s=CONTROL_PERIOD_S)


## Optional: quick offline simulation

If the training environment class `H2ProductionEnv` is importable (from your training script) and `hydrogen_data.csv` is available locally, you can run a short rollout to verify the policy before touching hardware.

In [ ]:
try:
    from hydrogen_env import H2ProductionEnv, register_env_if_needed, ENV_ID, resolve_csv_path
    import gymnasium as gym
    have_env = True
except Exception as exc:  # noqa: BLE001
    have_env = False
    print(f"Skipping simulation: could not import H2ProductionEnv ({exc}).")

if have_env:
    register_env_if_needed(max_episode_steps=300, force=True)
    data_csv = resolve_csv_path("hydrogen_data.csv")

    env = gym.make(
        ENV_ID,
        csv_path=data_csv,
        measurement_noise_std=0.03,
        randomize_degradation=False,
    )

    obs, info = env.reset(seed=0)
    controller.reset(getattr(env, "voltage", 2.5), getattr(env, "temperature", 750.0))
    target_current = float(obs[1])

    sim_log: list[dict] = []
    for step in range(200):
        measured_current = float(env.current)
        action_delta, v_cmd, t_cmd, raw_action, entry = controller.step(
            measured_current=measured_current,
            target_current=target_current,
            timestamp=step,
        )
        obs, reward, terminated, truncated, info = env.step(action_delta)
        entry.update({
            "reward": float(reward),
            "env_voltage": float(env.voltage),
            "env_temperature": float(env.temperature),
        })
        sim_log.append(entry)
        if terminated or truncated:
            break

    env.close()
    sim_df = pd.DataFrame(sim_log)
    display(sim_df.head())
    if not sim_df.empty:
        ax = sim_df[["measured_current", "target_current"]].plot(title="Current tracking (simulation)")
        ax.set_ylabel("A")
        ax.grid(True)

        ax2 = sim_df[["voltage_command", "temperature_command"]].plot(title="Setpoints (simulation)")
        ax2.set_ylabel("V / degC")
        ax2.grid(True)
